# PointNet v2 : Adding Input T-Net

(architecture upgrade — v1 baseline + 3x3 spatial transformer)

## Environment, imports and reproducibility



In [1]:
import os
import time
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3d projection)
from torch.utils.data import Dataset, DataLoader

# Reproducibility — fixed seeds so re-running gives the same numbers
SEED = 1234
np.random.seed(SEED)
torch.manual_seed(SEED)

# Pick the best device available: CUDA -> MPS -> CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")

PyTorch version : 2.12.0
Device          : mps


## Loading the pre-sampled point clouds (cache from v1)

In [2]:
# Find the project root no matter where Jupyter is launched from
_cwd = os.path.abspath(os.getcwd())
PROJECT_ROOT = os.path.dirname(_cwd) if os.path.basename(_cwd) == "notebooks" else _cwd

DATA_DIR       = os.path.join(PROJECT_ROOT, "data", "ModelNet10")
CACHE_DIR      = os.path.join(PROJECT_ROOT, "data", "cache_1024pts")
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Class list (alphabetical = deterministic labels across runs and notebooks)
CLASSES = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}


class CachedModelNet10(Dataset):
    """Reads pre-sampled point clouds from the .npz cache built in v1 notebook."""
    def __init__(self, cache_path):
        data = np.load(cache_path)
        self.points = data["points"]   # (n, 1024, 3) float32
        self.labels = data["labels"]   # (n,) int64

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.from_numpy(self.points[idx]), int(self.labels[idx])


BATCH_SIZE = 32

train_cached = CachedModelNet10(os.path.join(CACHE_DIR, "train.npz"))
test_cached  = CachedModelNet10(os.path.join(CACHE_DIR, "test.npz"))

train_loader = DataLoader(train_cached, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, drop_last=True)
test_loader  = DataLoader(test_cached,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train : {len(train_cached)} samples, {len(train_loader)} batches")
print(f"Test  : {len(test_cached)} samples, {len(test_loader)} batches")
print(f"Classes ({len(CLASSES)}): {CLASSES}")

# Quick smoke test
pts, lbls = next(iter(train_loader))
print(f"\nOne batch -> points {tuple(pts.shape)}, labels {tuple(lbls.shape)}")

Train : 3991 samples, 124 batches
Test  : 908 samples, 29 batches
Classes (10): ['bathtub', 'bed', 'chair', 'desk', 'dresser', 'monitor', 'night_stand', 'sofa', 'table', 'toilet']

One batch -> points (32, 1024, 3), labels (32,)


## Defining the T-Net 

In [3]:
class TNet(nn.Module):
    """Spatial transformer for point clouds. Predicts a k×k affine matrix
    from the input via a mini-PointNet (shared MLP -> max-pool -> FC).
    Initialized to output identity so the model behaves like plain PointNet at init.
    """
    def __init__(self, k=3):
        super().__init__()
        self.k = k
        self.shared_mlp = nn.Sequential(
            nn.Conv1d(k,   64,   1), nn.BatchNorm1d(64),   nn.ReLU(),
            nn.Conv1d(64,  128,  1), nn.BatchNorm1d(128),  nn.ReLU(),
            nn.Conv1d(128, 1024, 1), nn.BatchNorm1d(1024), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Linear(512,  256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256,  k * k),
        )
        # Identity-init: zero the final weights, set bias to vec(I_k)
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)
        with torch.no_grad():
            self.head[-1].bias.copy_(torch.eye(k).flatten())

    def forward(self, x):
        # x: (B, N, k) -> (B, k, N) for Conv1d
        B = x.size(0)
        x = x.transpose(1, 2)
        x = self.shared_mlp(x)              # (B, 1024, N)
        x = torch.max(x, dim=2)[0]          # (B, 1024)
        x = self.head(x)                    # (B, k*k)
        return x.view(B, self.k, self.k)    # (B, k, k)


# Sanity check: at init, T-Net should output the identity matrix for any input
_tnet = TNet(k=3)
with torch.no_grad():
    T = _tnet(torch.randn(2, 1024, 3))
print("T-Net output at init (should be identity for each batch item):")
print(T[0].numpy().round(4))
print(f"\nT-Net parameter count: {sum(p.numel() for p in _tnet.parameters()):,}")

T-Net output at init (should be identity for each batch item):
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

T-Net parameter count: 803,081


## Defining PointNet v2 (v1 + the T-Net)

In [4]:
class PointNetV2(nn.Module):
    """PointNet v2: v1 + input T-Net (3×3 spatial transform of the input)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.input_tnet = TNet(k=3)
        self.shared_mlp = nn.Sequential(
            nn.Conv1d(3,   64,   1), nn.BatchNorm1d(64),   nn.ReLU(),
            nn.Conv1d(64,  64,   1), nn.BatchNorm1d(64),   nn.ReLU(),
            nn.Conv1d(64,  64,   1), nn.BatchNorm1d(64),   nn.ReLU(),
            nn.Conv1d(64,  128,  1), nn.BatchNorm1d(128),  nn.ReLU(),
            nn.Conv1d(128, 1024, 1), nn.BatchNorm1d(1024), nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Linear(512,  256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256,  num_classes),
        )

    def forward(self, x):
        # x: (B, N, 3) — apply the learned 3x3 transform first
        T = self.input_tnet(x)              # (B, 3, 3)
        x = torch.bmm(x, T)                 # (B, N, 3) — multiply each point by T
        x = x.transpose(1, 2)               # (B, 3, N)
        x = self.shared_mlp(x)              # (B, 1024, N)
        x = torch.max(x, dim=2)[0]          # (B, 1024) — global max-pool
        x = self.classifier(x)              # (B, num_classes)
        return x


# Instantiate and inspect
model = PointNetV2(num_classes=10)
n_tnet       = sum(p.numel() for p in model.input_tnet.parameters())
n_mlp        = sum(p.numel() for p in model.shared_mlp.parameters())
n_classifier = sum(p.numel() for p in model.classifier.parameters())
n_total      = sum(p.numel() for p in model.parameters())

print("Model: PointNet v2 (v1 + input T-Net)\n")
print("Parameter count by block:")
print(f"  input T-Net (mini-PointNet) : {n_tnet:>9,}")
print(f"  shared per-point MLP        : {n_mlp:>9,}")
print(f"  classifier head             : {n_classifier:>9,}")
print(f"  {'-'*46}")
print(f"  total                       : {n_total:>9,}    ({n_total/811914:.2f}x v1)")

# Forward sanity on one real batch on the chosen device
model = model.to(device)
pts, lbls = next(iter(train_loader))
pts, lbls = pts.to(device), lbls.to(device)
with torch.no_grad():
    out = model(pts)
print(f"\nForward sanity:")
print(f"  input  shape : {tuple(pts.shape)}")
print(f"  output shape : {tuple(out.shape)}  (expected (32, 10))")
print(f"  finite?      : {torch.isfinite(out).all().item()}")

Model: PointNet v2 (v1 + input T-Net)

Parameter count by block:
  input T-Net (mini-PointNet) :   803,081
  shared per-point MLP        :   151,680
  classifier head             :   660,234
  ----------------------------------------------
  total                       : 1,614,995    (1.99x v1)

Forward sanity:
  input  shape : (32, 1024, 3)
  output shape : (32, 10)  (expected (32, 10))
  finite?      : True
